# Demo da API — LSTM PETR4

Consome a API de previsão em produção como um cliente HTTP qualquer (apenas `requests`).

Pré-requisitos:
- API no ar (URL na célula de setup) ou rodando local em `http://localhost:8000`
- Conexão com internet (o yfinance busca dados ao vivo)

Roda em Google Colab ou Jupyter local.

## 0. Setup

In [ ]:
# No Colab descomente:
# !pip install -q requests yfinance pandas matplotlib

import time
import requests
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Troque pela URL do seu deploy Render
API = 'https://tech-challenge-4-api-o61w.onrender.com'
# API = 'http://localhost:8000'  # se for testar local

print('Target:', API)

## 1. Health check

In [ ]:
r = requests.get(f'{API}/health', timeout=60)
print('HTTP', r.status_code, r.json())

## 2. Metadata do modelo

In [ ]:
info = requests.get(f'{API}/').json()
print('Ticker:        ', info['ticker'])
print('Janela LSTM:   ', info['window'], 'dias')
print('Treinado em:   ', info['trained_at'])
print('Framework:     ', info['framework'], info['framework_version'])
print()
print('Métricas TESTE (dados nunca vistos no treino):')
for k, v in info['metrics_test'].items():
    suffix = '%' if k == 'mape' else ' (R$)'
    print(f'  {k.upper():5s} = {v:.4f}{suffix}')

## 3. Predição imediata (1 dia à frente)

In [ ]:
r = requests.get(f'{API}/predict/next', timeout=60)
data = r.json()
print('Resposta da API:')
for k, v in data.items():
    print(f'  {k}: {v}')
    
# Comparação com o último close real
ultimo = yf.download('PETR4.SA', period='5d', progress=False, auto_adjust=False)['Close'].dropna()
if hasattr(ultimo.columns, 'get_level_values'):
    ultimo.columns = ultimo.columns.get_level_values(0)
    ultimo = ultimo.iloc[:, 0]
print(f'\nÚltimo close real: R$ {float(ultimo.iloc[-1]):.2f}')
print(f'Predição API:      R$ {data["prediction"]:.2f}')
print(f'Delta esperado:    R$ {data["prediction"] - float(ultimo.iloc[-1]):+.2f}')

## 4. POST /predict — passando os dados explicitamente

In [ ]:
# 4a) Happy path: 60 fechamentos reais de PETR4
df = yf.download('PETR4.SA', period='4mo', progress=False, auto_adjust=False)
if hasattr(df.columns, 'get_level_values'):
    df.columns = df.columns.get_level_values(0)
closes = df['Close'].dropna().tail(60).tolist()

r = requests.post(f'{API}/predict', json={'closes': closes}, timeout=60)
print('HTTP', r.status_code)
print(r.json())

In [ ]:
# 4b) Validação Pydantic — poucos closes
r = requests.post(f'{API}/predict', json={'closes': [30.0] * 10}, timeout=60)
print('HTTP', r.status_code, '— esperado 422')
print(r.json())

In [ ]:
# 4c) Validação Pydantic — valor inválido
r = requests.post(f'{API}/predict', json={'closes': [30.0] * 60 + [-1.0]}, timeout=60)
print('HTTP', r.status_code, '— esperado 422')
print(r.json())

## 5. Forecast multi-step + plot

In [ ]:
horizon = 10
r = requests.get(f'{API}/predict/forecast', params={'horizon': horizon}, timeout=60)
fc = r.json()['predictions']

# Plot histórico real + forecast
hist = yf.download('PETR4.SA', period='6mo', progress=False, auto_adjust=False)['Close'].dropna()
if hasattr(hist.columns, 'get_level_values'):
    hist.columns = hist.columns.get_level_values(0)
    hist = hist.iloc[:, 0]
future_idx = pd.bdate_range(start=hist.index[-1] + pd.Timedelta(days=1), periods=horizon)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(hist.index, hist.values, lw=1.0, color='#1f77b4', label='histórico real')
ax.plot(future_idx, fc, lw=1.5, color='#d62728', marker='o', label=f'forecast LSTM ({horizon}d)')
ax.axvline(hist.index[-1], color='gray', ls='--', alpha=0.5)
ax.set_title('PETR4.SA — histórico real + forecast da API')
ax.set_ylabel('R$')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for i, p in enumerate(fc, 1):
    print(f'  D+{i:2d}: R$ {p:.2f}')

## 6. Backtest visual — predizer cada dia da última semana

In [ ]:
df = yf.download('PETR4.SA', period='1y', progress=False, auto_adjust=False)
if hasattr(df.columns, 'get_level_values'):
    df.columns = df.columns.get_level_values(0)
closes = df['Close'].dropna()

n_days = 7
preds, reals, dates = [], [], []
for i in range(n_days, 0, -1):
    window = closes.iloc[-(60 + i):-i].tolist()
    real = float(closes.iloc[-i])
    r = requests.post(f'{API}/predict', json={'closes': window}, timeout=60)
    pred = r.json()['prediction']
    preds.append(pred); reals.append(real); dates.append(closes.index[-i])

errs = [abs(p - r) for p, r in zip(preds, reals)]
mape = np.mean([abs(p - r) / r for p, r in zip(preds, reals)]) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(dates, reals, lw=2, marker='o', label='real')
ax.plot(dates, preds, lw=2, marker='s', label='predito (API)', color='#d62728')
ax.set_title(f'Backtest dos últimos {n_days} pregões — MAPE {mape:.2f}%')
ax.set_ylabel('R$'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

tabela = pd.DataFrame({'data': [d.strftime('%Y-%m-%d') for d in dates], 'real': reals, 'predito': preds, 'erro_abs': errs})
print(tabela.to_string(index=False))

## 7. Latência — stress test leve

In [ ]:
n = 30
lat = []
for _ in range(n):
    t0 = time.perf_counter()
    requests.post(f'{API}/predict', json={'closes': closes.tail(60).tolist()}, timeout=60)
    lat.append((time.perf_counter() - t0) * 1000)

lat = np.array(lat)
print(f'N = {n}')
print(f'min  = {lat.min():.1f} ms')
print(f'p50  = {np.percentile(lat, 50):.1f} ms')
print(f'p95  = {np.percentile(lat, 95):.1f} ms')
print(f'p99  = {np.percentile(lat, 99):.1f} ms')
print(f'max  = {lat.max():.1f} ms')

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(lat, bins=20, color='#2ca02c', alpha=0.7, edgecolor='black')
ax.axvline(np.percentile(lat, 50), color='blue', ls='--', label='p50')
ax.axvline(np.percentile(lat, 95), color='red', ls='--', label='p95')
ax.set_xlabel('latência (ms)'); ax.set_ylabel('count')
ax.set_title(f'Distribuição de latência — {n} requests POST /predict')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Lendo o `/metrics` (Prometheus exposition format)

In [ ]:
raw = requests.get(f'{API}/metrics', timeout=60).text

print('=== Métricas customizadas ===\n')
for line in raw.splitlines():
    if line.startswith('#') or not line.strip():
        continue
    if any(k in line for k in (
        'http_requests_total',
        'model_predictions_total',
        'model_input_last_close',
        'model_prediction_latency_seconds_count',
        'model_prediction_latency_seconds_sum',
    )):
        print(line)

In [ ]:
# Bonus: parsing simples do http_requests_total em DataFrame
import re

rows = []
pat = re.compile(r'http_requests_total\{([^}]+)\}\s+(\d+\.?\d*)')
for m in pat.finditer(raw):
    labels = dict(re.findall(r'(\w+)="([^"]+)"', m.group(1)))
    labels['count'] = float(m.group(2))
    rows.append(labels)

if rows:
    pd.DataFrame(rows).sort_values('count', ascending=False)

## Links

- Repositório: https://github.com/gbsander/tech_challenge_4
- API: https://tech-challenge-4-api-o61w.onrender.com
- Swagger: https://tech-challenge-4-api-o61w.onrender.com/docs
- Métricas: https://tech-challenge-4-api-o61w.onrender.com/metrics